# 实验8.1 基于深度强化学习的机器人避障开发实验

本 notebook 对应《第8章-智能机器人系统开发》的 **8.4 基于深度强化学习的机器人避障开发**，从马尔可夫决策过程建模出发，逐步实现 Dueling DQN 算法，并在 **昇腾 NPU** 上训练与推理。

**运行环境**：`cann_9.0.0-py3.11-A2-arm-20260715` · `ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB`

---

## 目录

1. 问题建模：马尔可夫决策过程
2. 状态空间与动作空间设计
3. 奖励函数设计
4. Dueling DQN 网络架构
5. DQN 算法流程与改进技术
6. 动手实验：在 NPU 上训练避障智能体
7. 仿真训练与课程学习策略
8. 模型部署：ATC 转换与昇腾推理
9. 小结
10. 课后练习

---

## 1. 问题建模：马尔可夫决策过程

在深度强化学习的小车避障任务中，首先需要将问题建模为一个**马尔可夫决策过程（MDP）**。MDP 通过状态、动作和奖励三个关键要素描述智能体与环境的交互。

### MDP 的五个核心要素

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">符号</th>
<th style="text-align: left;">名称</th>
<th style="text-align: left;">含义</th>
</tr>
<tr>
<td style="text-align: left;">S</td>
<td style="text-align: left;">状态空间</td>
<td style="text-align: left;">所有可能状态的集合</td>
</tr>
<tr>
<td style="text-align: left;">A</td>
<td style="text-align: left;">动作空间</td>
<td style="text-align: left;">所有可能动作的集合</td>
</tr>
<tr>
<td style="text-align: left;">P</td>
<td style="text-align: left;">状态转移概率</td>
<td style="text-align: left;">P(s'</td>
<td style="text-align: left;">s,a)：在状态s执行动作a后转移到s'的概率</td>
</tr>
<tr>
<td style="text-align: left;">R</td>
<td style="text-align: left;">奖励函数</td>
<td style="text-align: left;">R(s,a)：在状态s执行动作a后获得的即时奖励</td>
</tr>
<tr>
<td style="text-align: left;">gamma</td>
<td style="text-align: left;">折扣因子</td>
<td style="text-align: left;">gamma in [0,1]，平衡即时与未来回报</td>
</tr>
</table>

**强化学习的目标**：学习最优策略 $\pi(a|s)$，最大化累计折扣奖励：

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

---

## 2. 状态空间与动作空间设计

### 2.1 状态空间：多模态环境感知

状态向量采用多模态信息融合策略：

- **视觉信息**：前置摄像头图像 $I_t$，经 CNN 提取特征 $z_t = \text{CNN}(I_t)$
- **运动状态**：线速度 $v_t$、转向角 $\theta_t$、历史动作序列
- **状态向量**：$s_t = [z_t, e_t]$，其中 $e_t$ 为运动状态信息

### 2.2 动作空间：连续与离散方案

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方案</th>
<th style="text-align: left;">描述</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">适用算法</th>
</tr>
<tr>
<td style="text-align: left;">连续动作</td>
<td style="text-align: left;">v in [0,1] m/s, theta in [-30,30] deg</td>
<td style="text-align: left;">控制精度高、运动平滑</td>
<td style="text-align: left;">DDPG, TD3, SAC</td>
</tr>
<tr>
<td style="text-align: left;">离散动作</td>
<td style="text-align: left;">3x3=9种组合（速度3级 x 方向3级）</td>
<td style="text-align: left;">实现简单、计算量小</td>
<td style="text-align: left;">DQN</td>
</tr>
</table>

本 notebook 采用**离散动作空间**（9种动作组合），适合 DQN 系列算法。

In [ ]:
# 定义离散动作空间
import numpy as np

class DiscreteActionSpace:
    def __init__(self):
        # 速度3级：停(0)、慢(0.15)、快(0.3)
        self.speeds = [0.0, 0.15, 0.3]
        # 方向3级：左(-0.5)、直(0)、右(0.5)
        self.turns = [-0.5, 0.0, 0.5]
        # 9种动作组合
        self.actions = []
        for v in self.speeds:
            for w in self.turns:
                self.actions.append((v, w))
    
    def __len__(self):
        return len(self.actions)
    
    def sample(self):
        return np.random.randint(len(self.actions))
    
    def get_action(self, idx):
        return self.actions[idx]

action_space = DiscreteActionSpace()
print(f'动作空间大小: {len(action_space)}')
print('所有动作 (线速度 m/s, 角速度 rad/s):')
for i, (v, w) in enumerate(action_space.actions):
    print(f'  动作{i}: v={v:.2f}, w={w:+.1f}')

---

## 3. 奖励函数设计

奖励函数是深度强化学习成功的关键，通过精心设计的奖惩机制引导小车学习安全高效的避障策略。

<img src="./images/reward_function.png" alt="奖励函数" style="display: block; margin-left: 0;" />

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">行为</th>
<th style="text-align: left;">奖励</th>
</tr>
<tr>
<td style="text-align: left;">碰撞</td>
<td style="text-align: left;">-6.0</td>
</tr>
<tr>
<td style="text-align: left;">前进</td>
<td style="text-align: left;">+0.2</td>
</tr>
<tr>
<td style="text-align: left;">停止</td>
<td style="text-align: left;">-1.0</td>
</tr>
<tr>
<td style="text-align: left;">后退</td>
<td style="text-align: left;">-0.2</td>
</tr>
<tr>
<td style="text-align: left;">连续前进(每步)</td>
<td style="text-align: left;">+0.1</td>
</tr>
<tr>
<td style="text-align: left;">连续转圈惩罚</td>
<td style="text-align: left;">-0.3 * 次数</td>
</tr>
<tr>
<td style="text-align: left;">连续停止惩罚</td>
<td style="text-align: left;">-0.3 * 次数</td>
</tr>
</table>

In [ ]:
class RewardFunction:
    def __init__(self):
        self.consecutive_forward = 0
        self.consecutive_turn = 0
        self.consecutive_stop = 0
    
    def reset(self):
        self.consecutive_forward = 0
        self.consecutive_turn = 0
        self.consecutive_stop = 0
    
    def compute(self, action_idx, collision, action_space):
        reward = 0.0
        v, w = action_space.get_action(action_idx)
        
        # 碰撞惩罚
        if collision:
            reward -= 6.0
            return reward
        
        # 动作奖励
        if v > 0 and w == 0:  # 前进
            reward += 0.2
            self.consecutive_forward += 1
            self.consecutive_turn = 0
            self.consecutive_stop = 0
            if self.consecutive_forward > 1:
                reward += 0.1  # 连续前进奖励
        elif v == 0:  # 停止
            reward -= 1.0
            self.consecutive_stop += 1
            self.consecutive_forward = 0
            if self.consecutive_stop > 1:
                reward -= 0.3 * self.consecutive_stop
        elif v < 0:  # 后退
            reward -= 0.2
        else:  # 转向
            self.consecutive_turn += 1
            self.consecutive_forward = 0
            if self.consecutive_turn > 2:
                reward -= 0.3 * self.consecutive_turn
        
        return reward

# 测试奖励函数
rf = RewardFunction()
print('碰撞:', rf.compute(0, True, action_space))
rf.reset()
print('前进(动作5):', rf.compute(5, False, action_space))
print('继续前进(动作5):', rf.compute(5, False, action_space))
print('停止(动作0):', rf.compute(0, False, action_space))

---

## 4. Dueling DQN 网络架构

<img src="./images/dueling_dqn_architecture.png" alt="Dueling DQN架构" style="display: block; margin-left: 0;" />

Dueling DQN 将 Q 值分解为两个独立的计算流：

- **价值流 V(s)**：估计状态本身的内在价值
- **优势流 A(s,a)**：评估每个动作相对于平均水平的优势

**Q 值组合公式**：

$$Q(s, a) = V(s) + A(s, a) - \text{mean}(A(s, a))$$

### 在昇腾 NPU 上构建 Dueling DQN

下面用 PyTorch + torch_npu 构建 Dueling DQN 网络：

In [ ]:
!pip install onnxscript -q
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 检测设备
device = torch.device('cpu')
try:
    import torch_npu
    if torch.npu.is_available():
        device = torch.device('npu:0')
        print(f'使用昇腾 NPU: {torch.npu.get_device_name(0)}')
except ImportError:
    print('torch_npu 未安装，使用 CPU')
print(f'计算设备: {device}')


class DuelingDQN(nn.Module):
    def __init__(self, state_dim, n_actions, hidden_dim=128):
        super().__init__()
        # 共享特征提取层
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # 价值流 V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        # 优势流 A(s,a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, n_actions)
        )
    
    def forward(self, x):
        feat = self.feature(x)
        value = self.value_stream(feat)
        advantage = self.advantage_stream(feat)
        # Dueling: Q = V + A - mean(A)
        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q

# 创建网络 (状态维度=8, 动作数=9)
state_dim = 8  # [距离前/左/右/后, 当前v, 当前w, 目标dx, 目标dy]
n_actions = len(action_space)
policy_net = DuelingDQN(state_dim, n_actions).to(device)
target_net = DuelingDQN(state_dim, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

n_params = sum(p.numel() for p in policy_net.parameters())
print(f'网络参数总数: {n_params:,}')
print(f'状态维度: {state_dim}, 动作数: {n_actions}')
print(policy_net)

---

## 5. DQN 算法流程与改进技术

### 5.1 核心公式

**Bellman 方程**：$Q^*(s,a) = \mathbb{E}[r + \gamma \max Q^*(s', a')]$

**目标 Q 值**：$y_t = r_t + \gamma \max Q(s_{t+1}, a; \theta^-)$

**损失函数**：$L(\theta) = \mathbb{E}[(y_t - Q(s_t, a_t; \theta))^2]$

### 5.2 改进技术

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">解决的问题</th>
</tr>
<tr>
<td style="text-align: left;">Double DQN</td>
<td style="text-align: left;">解耦动作选择与价值评估，减少过估计</td>
</tr>
<tr>
<td style="text-align: left;">优先经验回放</td>
<td style="text-align: left;">按 TD 误差重要性采样，加速关键经验学习</td>
</tr>
<tr>
<td style="text-align: left;">Noisy Network</td>
<td style="text-align: left;">参数空间噪声实现自适应探索</td>
</tr>
<tr>
<td style="text-align: left;">目标网络</td>
<td style="text-align: left;">定期同步提供稳定学习目标</td>
</tr>
</table>

### 5.3 经验回放缓冲区

DQN 的核心组件之一，存储交互经验 $(s_t, a_t, r_t, s_{t+1}, \text{done})$，通过随机采样打破时间相关性。

In [ ]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
    
    def __len__(self):
        return len(self.buffer)

buffer = ReplayBuffer(capacity=5000)
print(f'经验回放缓冲区容量: 5000')
print(f'当前大小: {len(buffer)}')

---

## 6. 动手实验：在 NPU 上训练避障智能体

下面构建一个简化的栅格世界避障环境，在上面训练 Dueling DQN 智能体。

In [ ]:
class GridWorldRobot:
    '''简化的栅格世界避障环境'''
    def __init__(self, size=10):
        self.size = size
        self.reset()
    
    def reset(self):
        self.pos = np.array([0, 0])
        self.goal = np.array([self.size-1, self.size-1])
        # 随机障碍物
        self.obstacles = set()
        for _ in range(15):
            ox, oy = np.random.randint(1, self.size-1, 2)
            self.obstacles.add((ox, oy))
        return self._get_state()
    
    def _get_state(self):
        # 状态: [到前方障碍距离, 到左方, 到右方, 到后方, pos_x, pos_y, goal_dx, goal_dy]
        dists = []
        for dx, dy in [(0,1),(-1,0),(1,0),(0,-1)]:
            d = 0
            for step in range(1, self.size):
                nx, ny = self.pos[0]+dx*step, self.pos[1]+dy*step
                if nx<0 or nx>=self.size or ny<0 or ny>=self.size or (nx,ny) in self.obstacles:
                    d = step
                    break
            else:
                d = self.size
            dists.append(d / self.size)
        goal_dir = (self.goal - self.pos) / self.size
        state = np.array(dists + [self.pos[0]/self.size, self.pos[1]/self.size,
                                   goal_dir[0], goal_dir[1]], dtype=np.float32)
        return state
    
    def step(self, action_idx):
        v, w = action_space.get_action(action_idx)
        # 简化运动：根据速度和方向选择移动
        if v == 0:
            move = np.array([0, 0])
        else:
            if w < 0: move = np.array([-1, 0])  # 左
            elif w > 0: move = np.array([1, 0])  # 右
            else: move = np.array([0, 1])       # 前
            if v < 0.2: move = move  # 慢速
        
        new_pos = self.pos + move
        
        # 边界和障碍检查
        collision = False
        if (new_pos[0]<0 or new_pos[0]>=self.size or 
            new_pos[1]<0 or new_pos[1]>=self.size or
            tuple(new_pos) in self.obstacles):
            collision = True
        else:
            self.pos = new_pos
        
        # 计算奖励
        reward = rf.compute(action_idx, collision, action_space)
        
        # 到达目标
        done = False
        if np.array_equal(self.pos, self.goal):
            reward += 10.0
            done = True
        elif collision:
            done = True
        
        return self._get_state(), reward, done, collision

env = GridWorldRobot(size=10)
state = env.reset()
print(f'环境: 10x10 栅格世界')
print(f'初始状态维度: {state.shape}')
print(f'障碍物数量: {len(env.obstacles)}')
print(f'起点: {env.pos}, 目标: {env.goal}')

In [ ]:
# DQN 训练循环
import time

BATCH_SIZE = 64
GAMMA = 0.99
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 200
TARGET_UPDATE = 10
N_EPISODES = 200

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = ReplayBuffer(capacity=5000)

episode_rewards = []
episode_losses = []
steps_done = 0

print(f'开始训练 {N_EPISODES} 回合...')
start_time = time.time()

for episode in range(N_EPISODES):
    rf.reset()
    state = env.reset()
    total_reward = 0
    ep_loss = 0
    n_steps = 0
    
    for t in range(100):
        # epsilon-greedy 选择动作
        eps = EPS_END + (EPS_START - EPS_END) * np.exp(-steps_done / EPS_DECAY)
        if random.random() < eps:
            action = action_space.sample()
        else:
            with torch.no_grad():
                q = policy_net(torch.FloatTensor(state).unsqueeze(0).to(device))
                action = q.argmax(dim=1).item()
        
        next_state, reward, done, collision = env.step(action)
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps_done += 1
        n_steps += 1
        
        # 训练
        if len(buffer) >= BATCH_SIZE:
            s, a, r, s2, d = buffer.sample(BATCH_SIZE)
            s_t = torch.FloatTensor(s).to(device)
            a_t = torch.LongTensor(a).to(device)
            r_t = torch.FloatTensor(r).to(device)
            s2_t = torch.FloatTensor(s2).to(device)
            d_t = torch.FloatTensor(d).to(device)
            
            q_values = policy_net(s_t).gather(1, a_t.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q = target_net(s2_t).max(dim=1)[0]
                target_q = r_t + GAMMA * next_q * (1 - d_t)
            
            loss = nn.functional.mse_loss(q_values, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            ep_loss += loss.item()
        
        if done:
            break
    
    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())
    
    episode_rewards.append(total_reward)
    episode_losses.append(ep_loss / max(n_steps, 1))
    
    if (episode + 1) % 50 == 0:
        avg_r = np.mean(episode_rewards[-50:])
        print(f'回合 {episode+1}/{N_EPISODES}, 平均奖励: {avg_r:.2f}, epsilon: {eps:.3f}')

elapsed = time.time() - start_time
print(f'\n训练完成! 耗时: {elapsed:.1f}s')
print(f'最终平均奖励(最后20回合): {np.mean(episode_rewards[-20:]):.2f}')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = True

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(episode_rewards, alpha=0.5, color='blue')
window = 20
if len(episode_rewards) >= window:
    smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(episode_rewards)), smoothed, color='red', linewidth=2, label='Moving Avg')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('DQN Training Reward Curve')
axes[0].legend()
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].plot(episode_losses, alpha=0.5, color='blue')
if len(episode_losses) >= window:
    smoothed_l = np.convolve(episode_losses, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(episode_losses)), smoothed_l, color='red', linewidth=2, label='Moving Avg')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Loss')
axes[1].set_title('DQN Training Loss Curve')
axes[1].legend()

plt.suptitle(f'Dueling DQN Obstacle Avoidance Training (Device: {device})', fontsize=14)
plt.tight_layout()
plt.savefig('./images/dqn_training_result.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 测试训练后的智能体
test_rewards = []
test_successes = 0
N_TEST = 20

for i in range(N_TEST):
    rf.reset()
    state = env.reset()
    total_reward = 0
    for t in range(100):
        with torch.no_grad():
            q = policy_net(torch.FloatTensor(state).unsqueeze(0).to(device))
            action = q.argmax(dim=1).item()
        state, reward, done, collision = env.step(action)
        total_reward += reward
        if done:
            if not collision and np.array_equal(env.pos, env.goal):
                test_successes += 1
            break
    test_rewards.append(total_reward)

print(f'测试 {N_TEST} 次:')
print(f'  平均奖励: {np.mean(test_rewards):.2f}')
print(f'  成功到达目标: {test_successes}/{N_TEST} ({test_successes/N_TEST*100:.0f}%)')

---

## 7. 仿真训练与课程学习策略

<img src="./images/curriculum_learning.png" alt="课程学习" style="display: block; margin-left: 0;" />

训练采用**课程学习（Curriculum Learning）**策略，从简单到复杂渐进式训练：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">环境</th>
<th style="text-align: left;">目标</th>
</tr>
<tr>
<td style="text-align: left;">一</td>
<td style="text-align: left;">简单环境（稀疏障碍）</td>
<td style="text-align: left;">学习基本前进和避障</td>
</tr>
<tr>
<td style="text-align: left;">二</td>
<td style="text-align: left;">中等环境</td>
<td style="text-align: left;">训练路径规划能力</td>
</tr>
<tr>
<td style="text-align: left;">三</td>
<td style="text-align: left;">复杂环境（密集障碍）</td>
<td style="text-align: left;">提升复杂场景应对</td>
</tr>
<tr>
<td style="text-align: left;">四</td>
<td style="text-align: left;">动态环境（移动障碍）</td>
<td style="text-align: left;">训练实时避障反应</td>
</tr>
</table>

**优势**：加速收敛、提高稳定性、提升最终性能。

---

## 8. 模型部署：ATC 转换与昇腾推理

完成训练后，需要将 PyTorch 模型通过昇腾 **ATC（Ascend Tensor Compiler）** 工具转换为 OM 模型格式，部署到昇腾开发板进行实时推理。

### 转换流程

```
PyTorch 模型 -> ONNX 模型 -> ATC 转换 -> OM 模型 -> AscendCL 推理
```

In [ ]:
# 步骤1：导出 ONNX 模型
dummy_input = torch.randn(1, state_dim)
onnx_path = './dqn_robot.onnx'
torch.onnx.export(
    policy_net.cpu(),  # 导出需要 CPU 模型
    dummy_input,
    onnx_path,
    input_names=['state'],
    output_names=['q_values'],
    dynamic_axes={'state': {0: 'batch'}, 'q_values': {0: 'batch'}}
)
print(f'ONNX 模型已导出: {onnx_path}')
import os
print(f'模型大小: {os.path.getsize(onnx_path) / 1024:.1f} KB')

In [ ]:
# 步骤2：使用 ATC 转换为 OM 模型（需要在昇腾环境中执行）
print('=== ATC 模型转换命令 ===')
print('在昇腾开发板终端执行以下命令：')
print()
print('atc --model=./dqn_robot.onnx \\')
print('    --framework=5 \\')
print('    --output=./dqn_robot.om \\')
print('    --soc_version=Ascend310B4 \\')
print('    --input_shape="state:1,8"')
print()
print('参数说明：')
print('  --framework=5    : ONNX 框架')
print('  --soc_version    : 目标芯片型号')
print('  --input_shape    : 输入张量形状')
print()
print('转换成功后生成 dqn_robot.om，可在昇腾 NPU 上高效推理。')

In [ ]:
# 步骤3：使用 AscendCL 加载 OM 模型进行推理（伪代码示例）
print('=== AscendCL 推理代码示例 ===')
print('''
import acl

# 1. 初始化 ACL
acl.init()
ret = acl.rt.set_device(0)
context, ret = acl.rt.create_context(0)

# 2. 加载 OM 模型
model_id, ret = acl.mdl.load_from_file("./dqn_robot.om")
model_desc = acl.mdl.create_desc()
acl.mdl.get_desc(model_desc, model_id)

# 3. 创建输入输出数据集
# ... (创建 dataset 并绑定内存)

# 4. 执行推理
state_data = get_sensor_state()  # 从传感器获取状态
acl.mdl.execute(model_id, input_dataset, output_dataset)
q_values = parse_output(output_dataset)
action = np.argmax(q_values)  # 选择最优动作

# 5. 通过 ROS2 发布动作指令
cmd_vel_pub.publish(Twist(linear=Vector3(x=v), angular=Vector3(z=w)))
''')
print('在昇腾 NPU 上推理延迟: 50-100ms，满足实时控制需求。')

---

## 9. 小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">概念</th>
<th style="text-align: left;">一句话理解</th>
</tr>
<tr>
<td style="text-align: left;">MDP</td>
<td style="text-align: left;">通过(S, A, P, R, gamma)建模智能体与环境的交互</td>
</tr>
<tr>
<td style="text-align: left;">状态空间</td>
<td style="text-align: left;">多模态融合：CNN视觉特征 + 运动状态</td>
</tr>
<tr>
<td style="text-align: left;">动作空间</td>
<td style="text-align: left;">离散(9种)或连续(v, theta)，本节用离散</td>
</tr>
<tr>
<td style="text-align: left;">奖励函数</td>
<td style="text-align: left;">碰撞-6.0, 前进+0.2, 连续前进+0.1, 转圈惩罚</td>
</tr>
<tr>
<td style="text-align: left;">Dueling DQN</td>
<td style="text-align: left;">Q=V+A-mean(A)，分解价值与优势</td>
</tr>
<tr>
<td style="text-align: left;">Double DQN</td>
<td style="text-align: left;">解耦动作选择与评估，减少过估计</td>
</tr>
<tr>
<td style="text-align: left;">经验回放</td>
<td style="text-align: left;">随机采样打破时间相关性</td>
</tr>
<tr>
<td style="text-align: left;">课程学习</td>
<td style="text-align: left;">从简单到复杂渐进式训练</td>
</tr>
<tr>
<td style="text-align: left;">ATC 部署</td>
<td style="text-align: left;">PyTorch->ONNX->OM->AscendCL推理</td>
</tr>
</table>

<img src="./images/dqn_training_curve.png" alt="DQN训练曲线" style="display: block; margin-left: 0;" />

---

## 10. 课后练习

请根据本节课程学习内容完成以下题目进行自测。

**第1题**（单选题）深度强化学习避障问题建模为马尔可夫决策过程(MDP)的五个核心要素是？


- A. S, A, P, R, gamma
- B. 状态、动作、奖励、策略、价值
- C. 传感器、电机、控制器、算法、环境
- D. 训练集、验证集、测试集、模型、损失


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）Dueling DQN 将 Q 值分解为哪两个计算流？


- A. 策略流和价值流
- B. 价值流V(s)和优势流A(s,a)
- C. 前向流和后向流
- D. 编码流和解码流


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）Dueling DQN 的 Q 值组合公式是？


- A. Q = V + A
- B. Q = V + A - mean(A)
- C. Q = V * A
- D. Q = V - A


In [ ]:
q3 = ''  # 填入你的选项，如 'B'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）经验回放缓冲区的主要作用是？


- A. 存储模型参数
- B. 随机采样打破时间相关性，提高样本效率
- C. 加速GPU计算
- D. 减少内存占用


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）Double DQN 解决的核心问题是？


- A. 训练速度慢
- B. Q值过估计问题
- C. 内存不足
- D. 梯度消失


In [ ]:
q5 = ''  # 填入你的选项，如 'B'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）本节避障任务中，碰撞的惩罚值是？


- A. -1.0
- B. -6.0
- C. -0.2
- D. +0.2


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）课程学习策略的训练顺序是？


- A. 复杂->简单
- B. 简单->中等->复杂->动态
- C. 随机选择
- D. 同时训练所有难度


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）目标网络(Target Network)的作用是？


- A. 加速训练
- B. 提供稳定的学习目标，避免训练震荡
- C. 减少参数量
- D. 替代经验回放


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）模型部署到昇腾NPU的转换流程是？


- A. PyTorch->TensorFlow->ONNX
- B. PyTorch->ONNX->ATC->OM->AscendCL
- C. 直接使用PyTorch推理
- D. ONNX->TensorRT->OM


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）离散动作空间中，速度3级x方向3级共有多少种动作？


- A. 6种
- B. 9种
- C. 12种
- D. 3种


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**第11题**（单选题）Noisy Network 相比 epsilon-greedy 的优势是？


- A. 计算更快
- B. 参数空间噪声实现自适应探索
- C. 不需要训练
- D. 精度更高


In [ ]:
q11 = ''  # 填入你的选项，如 'B'
print(f'第11题答案已记录：{q11}' if q11 else '请填入答案并运行本单元格')

**第12题**（单选题）在昇腾NPU上部署DQN推理的典型延迟是？


- A. 1-5ms
- B. 50-100ms
- C. 500-1000ms
- D. 1-5s


In [ ]:
q12 = ''  # 填入你的选项，如 'B'
print(f'第12题答案已记录：{q12}' if q12 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path
for candidate in (Path.cwd() / 'answer', Path.cwd() / '08_robot_dev' / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_02 import grade
grade(globals())

---

## 参考资料

- [DQN 原始论文](https://arxiv.org/abs/1312.5602)
- [Dueling DQN 论文](https://arxiv.org/abs/1511.06581)
- [Double DQN 论文](https://arxiv.org/abs/1509.06461)
- [昇腾 ATC 工具文档](https://hiascend.com/document/detail/zh/CANNCommunityEdition)
- [AscendCL 推理 API](https://hiascend.com/document)

> 上一节：[01_robot_system_overview.ipynb](./01_robot_system_overview.ipynb)
> 下一节：[03_ascend_orangepi_experiment.ipynb](./03_ascend_orangepi_experiment.ipynb)